In [1]:
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from nltk.tokenize import word_tokenize
from itertools import combinations
import torch
import numpy as np
import random
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

import sys
sys.path.append('./tools')
from Matave import Matave

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [7]:
# Set up constant variables.
INPUT_FOLDER = 'data'
OUTPUT_FOLDER = 'output_explanations'
DEVICE = 0 if torch.cuda.is_available() else -1

# Make model variables.
MODEL_NAME = "nlpie/tiny-clinicalbert"
PEFT_HEAD = "../classifiers/peft/synth_lora_model_tinyclinicalbert"


LABEL_MAP = {
    0: "met",
    1: "unmet"
}

k_range = list(range(3, 20))
top_n = 10

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [4]:
# Get real data to work with.
all_data = {}
for folder in os.listdir(f'../{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'../{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'../{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder: 
                        daily_nurse = pd.read_excel(f'../{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes_{sub_sub_folder.split(' ')[0]}.xlsx')
                        daily_nurse = daily_nurse.dropna(subset=['Note', 'Date', 'Time'])
                        all_data[f"{sub_sub_folder.split(' ')[0]}"] = daily_nurse['Note'].dropna().values.tolist()

# Prepare dataset.
all_texts = []

for patient_id in all_data:
    all_texts.extend(all_data[patient_id])

print(f"Number of notes: {len(all_texts)}")

Number of notes: 12373


In [8]:
# Load model.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model_classif = PeftModel.from_pretrained(base_model, PEFT_HEAD)
tokenizer_classif = AutoTokenizer.from_pretrained(PEFT_HEAD)

Loading weights:   0%|          | 0/69 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpie/tiny-clinicalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [9]:
def predict_notes(notes, model, tokenizer, batch_size=16):
    model.eval()
    all_preds = []

    for i in range(0, len(notes), batch_size):
        batch = notes[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model(**inputs)

        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())

    return all_preds

predictions = predict_notes(all_texts, model_classif, tokenizer_classif)

In [22]:
all_df = pd.DataFrame({'notes': all_texts, 'labels': predictions})
met_texts = all_df[all_df['labels'] == 0]['notes'].values.tolist()
unmet_texts = all_df[all_df['labels'] == 1]['notes'].values.tolist()

In [23]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [24]:
def matave_compute(texts):
    # Run matave. 
    matave = Matave(texts)
    matave.fit(k_range = k_range)
    cluster_topics = [topic.split() for topic in matave.top_topic_words.values()]
    cluster_topics = [topic[:top_n] for topic in cluster_topics]

    # Prepare components for evaluation.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    # Evaluate using metrics.
    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}\nDiversity: {diversity}\nRedundancy: {redundancy}")
    
    matave.visualize()

    return matave, cluster_topics, {'coherence': coherence, 'diversity': diversity, 'redundancy': redundancy}

In [25]:
all_matave, all_cluster_topics, all_metrics = matave_compute(all_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/194 [00:00<?, ?it/s]

Coherence: 0.473493435850833
Diversity: 0.6714285714285714
Redundancy: 0.8380952380952381


In [26]:
met_matave, met_cluster_topics, met_metrics = matave_compute(met_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/169 [00:00<?, ?it/s]

Coherence: 0.4575776106701453
Diversity: 0.5666666666666667
Redundancy: 0.8878787878787879


In [28]:
unmet_matave, unmet_cluster_topics, unmet_metrics = matave_compute(unmet_texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Coherence: 0.521665697461072
Diversity: 0.5166666666666667
Redundancy: 0.9455555555555556
